# Dataset

In [ ]:
import torch
from src.options import opt_dict
from src.data.realcamvid_dataset import RealcamvidDataset

opt = opt_dict["wan2.1_t2v_1.3b_i2v"]
opt.load_da3_cam, opt.load_depth = True, True
dataset = RealcamvidDataset(opt, training=True)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True)

In [ ]:

data = next(iter(dataloader))
for k, v in data.items():
    print(f"{k}: {v.shape if isinstance(v, torch.Tensor) else v}")

import imageio.v2 as iio
from src.utils import *

I = 0

print(data["uid"][I])
iio.mimwrite("temp.mp4", tensor_to_video(data["image"][I]), fps=30)
print(data["C2W"][0, 0], data["fxfycxcy"][0, 0])

if "depth" in data:
    depths, C2W, fxfycxcy = data["depth"], data["C2W"], data["fxfycxcy"]
    print("Depth min & max:", depths[I].min(), depths[I].max())
    iio.mimwrite("temp_depth.mp4", tensor_to_video(colorize_depth(1./depths[I])), fps=30)

    xyzs = unproject_depth(depths[I:I+1, ...], C2W[I:I+1, ...], fxfycxcy[I:I+1, ...])[0]
    xyzs = normalize_among_last_dims(rearrange(xyzs, "f c h w -> c f h w"), num_dims=3)
    xyzs = rearrange(xyzs, "c f h w -> f c h w")
    iio.mimwrite("temp_xyzs.mp4", tensor_to_video(xyzs), fps=30)

    save_xyz_rgb_as_ply("temp.ply", xyzs, data["image"][I], ratio=0.02)

# Visualize DA3

In [ ]:
import imageio.v2 as iio
import numpy as np
from decord import VideoReader, cpu
import torchvision.transforms as tvT
from src.options import ROOT
from src.utils import *

x = np.load(f"{ROOT}/data/RealCam-Vid-DA3/RealEstate10K/test/dbFGT4L8aBw/651dda44a5c1e293.npz")
vr = VideoReader(str(f"{ROOT}/data/RealCam-Vid/RealEstate10K/test/dbFGT4L8aBw/651dda44a5c1e293.mp4"), ctx=cpu(0))
v = vr[:].asnumpy()

depths, confs, W2C, intrinsics = x["depth"], x["conf"], x["extrinsics"], x["intrinsics"]
print(depths.shape, confs.shape, W2C.shape, intrinsics.shape)

v = torch.from_numpy(v).permute(0, 3, 1, 2) / 255.
v = tvT.Resize((280, 504), tvT.InterpolationMode.BICUBIC)(v).clamp(0., 1.)
print(v.shape)


In [ ]:
depths = torch.from_numpy(depths).float()
confs = torch.from_numpy(confs).float()

W2C_ = torch.eye(4).unsqueeze(0).repeat(W2C.shape[0], 1, 1)
W2C_[:, :3, :4] = torch.from_numpy(W2C).float()
C2W = inverse_c2w(W2C_)
intrinsics[:, 0, 0] /= 504
intrinsics[:, 1, 1] /= 280
intrinsics[:, 0, 2] /= 504
intrinsics[:, 1, 2] /= 280
fxfycxcy = intrinsics_to_fxfycxcy(torch.from_numpy(intrinsics).float()[None, ...])[0]

In [ ]:
iio.mimwrite("temp.mp4", tensor_to_video(v), fps=30)
iio.mimwrite("temp_depth.mp4", tensor_to_video(colorize_depth(1./depths)), fps=30)
iio.mimwrite("temp_conf.mp4", tensor_to_video(colorize_depth(confs)), fps=30)

xyzs = unproject_depth(depths[None, ...], C2W[None, ...], fxfycxcy[None, ...])[0]
xyzs = normalize_among_last_dims(rearrange(xyzs, "f c h w -> c f h w"), num_dims=3)
xyzs = rearrange(xyzs, "c f h w -> f c h w")
iio.mimwrite("temp_xyzs.mp4", tensor_to_video(xyzs), fps=30)

save_xyz_rgb_as_ply("temp.ply", xyzs, v, ratio=0.02)

# Wan

In [ ]:
import torch
from src.options import opt_dict, ROOT
from src.models import MyEMAModel
from src.models import Wan

opt = opt_dict["wan2.1_t2v_1.3b_i2v"]
model = Wan(opt)

tag, iter = "wan_i2v_cc_81x512_tf", 2000

ema_path = f"{ROOT}/projects/VGGM/out/archive/{tag}/checkpoints/{iter:06d}/ema_states.pth"
ema_states = MyEMAModel(model.parameters())
ema_states.load_state_dict(torch.load(ema_path, map_location="cpu", weights_only=True))
ema_states.copy_to(model.parameters())

torch.save(model.plucker_embed.state_dict(), f"{tag}_{iter:06d}_plucker_embed.pth")
torch.save(model.diffusion.state_dict(), f"{tag}_{iter:06d}.pth")

# WanDA3

In [ ]:
import torch
from src.options import opt_dict
from src.models.networks.wan_wrapper import WanDiffusionDA3Wrapper

opt = opt_dict["wan2.1_t2v_1.3b_i2v"]
model = WanDiffusionDA3Wrapper(opt.wan_dir).to("cuda")

noisy_latents = torch.randn(2, 16, 21, 36, 64).to("cuda")
timesteps = torch.randint(0, 1000, (2,)).to("cuda")
text_embeddings = torch.randn(2, 100, 4096).to("cuda")

with torch.no_grad():
    outputs = model(noisy_latents, timesteps, text_embeddings)